# 00 — Data Exploration (EDA)

Analysis of both datasets before ML experiments.

In [26]:
import sys
from pathlib import Path

import matplotlib
matplotlib.use('Agg')  # headless-safe backend
import matplotlib.pyplot as plt

# Handle both local (run from Notebooks/) and Colab execution
ROOT = Path("..").resolve()
if not (ROOT / "models").exists():
    import subprocess
    subprocess.run(["git", "clone", "https://github.com/UrranQx/WaterMeterCV.git"], check=True)
    ROOT = Path("WaterMeterCV").resolve()

sys.path.insert(0, str(ROOT))

import pandas as pd
import cv2
import numpy as np
from models.data.unified_loader import load_water_meter_dataset, load_utility_meter_dataset
from models.utils.visualization import draw_roi_polygon, draw_digit_bboxes

print(f"ROOT: {ROOT}")

ROOT: C:\Users\alike\WaterMeterCV


In [27]:
wm_samples = load_water_meter_dataset(ROOT / "WaterMetricsDATA/waterMeterDataset/WaterMeters")
um_train = load_utility_meter_dataset(
    ROOT / "WaterMetricsDATA/utility-meter-reading-dataset-for-automatic-reading-yolo.v4i.yolov11",
    split="train",
)
um_valid = load_utility_meter_dataset(
    ROOT / "WaterMetricsDATA/utility-meter-reading-dataset-for-automatic-reading-yolo.v4i.yolov11",
    split="valid",
)
um_test = load_utility_meter_dataset(
    ROOT / "WaterMetricsDATA/utility-meter-reading-dataset-for-automatic-reading-yolo.v4i.yolov11",
    split="test",
)

print(f"WaterMeter: {len(wm_samples)} samples")
print(f"UtilityMeter: {len(um_train)} train, {len(um_valid)} valid, {len(um_test)} test")

WaterMeter: 1244 samples
UtilityMeter: 1552 train, 194 valid, 193 test


## Value Distribution (waterMeterDataset)

In [28]:
wm_values = [s.value for s in wm_samples if s.value is not None]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(wm_values, bins=50)
axes[0].set_title("waterMeterDataset — value distribution")
axes[0].set_xlabel("Meter reading")

digit_counts = [len(str(v).replace(".", "").replace("-", "")) for v in wm_values]
axes[1].hist(digit_counts, bins=range(1, 12))
axes[1].set_title("Number of digits per reading")
plt.tight_layout()
plt.savefig(ROOT / "results" / "eda_value_distribution.png", dpi=150)
plt.close()
print(f"Value range: {min(wm_values):.3f} — {max(wm_values):.3f}")
print(f"Mean: {sum(wm_values)/len(wm_values):.3f}")

Value range: 0.000 — 99999.539
Mean: 422.507


## ROI Polygon Samples (waterMeterDataset)

In [29]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, sample in zip(axes.flat, wm_samples[:8]):
    img = cv2.imread(str(sample.image_path))
    if img is None:
        ax.set_title("image not found")
        ax.axis("off")
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    if sample.roi_polygon:
        img = draw_roi_polygon(img, sample.roi_polygon)
    ax.imshow(img)
    ax.set_title(f"val={sample.value}")
    ax.axis("off")
plt.suptitle("waterMeterDataset — ROI polygons")
plt.tight_layout()
plt.savefig(ROOT / "results" / "eda_roi_samples.png", dpi=150)
plt.close()
print("ROI samples saved")

ROI samples saved


## Digit BBox Samples (utility-meter)

In [30]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, sample in zip(axes.flat, um_train[:8]):
    img = cv2.imread(str(sample.image_path))
    if img is None:
        ax.set_title("image not found")
        ax.axis("off")
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    if sample.digit_bboxes:
        img = draw_digit_bboxes(img, sample.digit_bboxes)
    ax.imshow(img)
    ax.set_title(f"val={sample.value}")
    ax.axis("off")
plt.suptitle("utility-meter — digit bounding boxes")
plt.tight_layout()
plt.savefig(ROOT / "results" / "eda_digit_bboxes.png", dpi=150)
plt.close()
print("Digit bbox samples saved")

Digit bbox samples saved


## Digit Class Distribution (utility-meter train)

In [31]:
from collections import Counter

all_digits = []
for s in um_train:
    if s.digit_bboxes:
        all_digits.extend([d[0] for d in s.digit_bboxes])

counts = Counter(all_digits)
plt.figure(figsize=(10, 5))
plt.bar([str(k) for k in sorted(counts.keys())], [counts[k] for k in sorted(counts.keys())])
plt.title("Digit class distribution (utility-meter train)")
plt.xlabel("Digit")
plt.ylabel("Count")
plt.savefig(ROOT / "results" / "eda_digit_distribution.png", dpi=150)
plt.close()
print(f"Total digit annotations: {len(all_digits)}")
print("Distribution:", dict(sorted(counts.items())))

Total digit annotations: 11755
Distribution: {0: 3886, 1: 1170, 2: 973, 3: 918, 4: 848, 5: 845, 6: 825, 7: 772, 8: 759, 9: 759}


## ROI Size Analysis (waterMeterDataset)

In [32]:
from shapely.geometry import Polygon

areas = []
for s in wm_samples:
    if s.roi_polygon and len(s.roi_polygon) >= 3:
        areas.append(Polygon(s.roi_polygon).area)

plt.figure(figsize=(10, 5))
plt.hist(areas, bins=30)
plt.title("ROI polygon area distribution (normalized coords)")
plt.xlabel("Area")
plt.ylabel("Count")
plt.savefig(ROOT / "results" / "eda_roi_areas.png", dpi=150)
plt.close()
print(f"Median ROI area: {sorted(areas)[len(areas)//2]:.4f}")
print(f"Area range: {min(areas):.4f} — {max(areas):.4f}")

Median ROI area: 0.0201
Area range: 0.0027 — 0.1500


## Duplicates & Cross-Dataset Overlap

In [35]:

import os
from collections import defaultdict

def file_size(s):
    try:
        return os.path.getsize(s.image_path)
    except OSError:
        return None

def dup_key(s):
    """Composite key: (value, file_size_bytes). None if either is unavailable."""
    v = s.value
    sz = file_size(s)
    if v is None or sz is None:
        return None
    return (v, sz)

def find_dup_groups(samples, key_fn):
    buckets = defaultdict(list)
    for s in samples:
        k = key_fn(s)
        if k is not None:
            buckets[k].append(s)
    return [g for g in buckets.values() if len(g) > 1]

def show_dups(groups, limit=10):
    if not groups:
        print("  no duplicates found")
        return
    total = sum(len(g) for g in groups)
    print(f"  {total} samples in {len(groups)} duplicate groups")
    for i, g in enumerate(groups[:limit]):
        key = dup_key(g[0])
        print(f"  group {i+1}  val={key[0]}  size={key[1]} bytes")
        for s in g:
            print(f"    {s.image_path}")
    if len(groups) > limit:
        print(f"  ... and {len(groups) - limit} more groups")

# ── waterMeterDataset ─────────────────────────────────────────────────────────
print("=" * 60)
print(f"waterMeterDataset  (N = {len(wm_samples)})")
print("=" * 60)
show_dups(find_dup_groups(wm_samples, dup_key))

# ── utility-meter (all splits) ────────────────────────────────────────────────
um_all = um_train + um_valid + um_test
print(f"\n{'=' * 60}")
print(f"utility-meter  train+valid+test  (N = {len(um_all)})")
print("=" * 60)
show_dups(find_dup_groups(um_all, dup_key))

# ── cross-dataset overlap ─────────────────────────────────────────────────────
print(f"\n{'=' * 60}")
print("Cross-dataset overlap  (value + file size)")
print("=" * 60)

wm_key_map = defaultdict(list)
for s in wm_samples:
    k = dup_key(s)
    if k is not None:
        wm_key_map[k].append(s)

cross_pairs = []
for s in um_all:
    k = dup_key(s)
    if k is not None and k in wm_key_map:
        for wm_s in wm_key_map[k]:
            cross_pairs.append((wm_s, s))

if not cross_pairs:
    print("  no cross-dataset matches found")
else:
    print(f"  {len(cross_pairs)} matching pairs")
    for wm_s, um_s in cross_pairs[:10]:
        k = dup_key(wm_s)
        print(f"  val={k[0]}  size={k[1]} bytes")
        print(f"    WM: {wm_s.image_path}")
        print(f"    UM: {um_s.image_path}")
    if len(cross_pairs) > 10:
        print(f"  ... and {len(cross_pairs) - 10} more pairs")


waterMeterDataset  (N = 1244)
  no duplicates found

utility-meter  train+valid+test  (N = 1939)
  no duplicates found

Cross-dataset overlap  (value + file size)
  no cross-dataset matches found


In [38]:
# Cross-dataset overlap by value only (exclude 0)
from collections import defaultdict

def value_key_nonzero(sample):
    if sample.value is None:
        return None
    value = round(float(sample.value), 3)
    # if abs(value) < 1e-9:
    #     return None
    return value

um_all = um_train + um_valid + um_test

wm_val_map = defaultdict(list)
for sample in wm_samples:
    key = value_key_nonzero(sample)
    if key is not None:
        wm_val_map[key].append(sample)

um_val_map = defaultdict(list)
for sample in um_all:
    key = value_key_nonzero(sample)
    if key is not None:
        um_val_map[key].append(sample)

common_values = sorted(set(wm_val_map.keys()) & set(um_val_map.keys()))

print("=" * 70)
print("Cross-dataset overlap (value only, excluding 0)")
print("=" * 70)

if not common_values:
    print("No overlaps found.")
else:
    print(f"Common values: {len(common_values)}")
    print(f"WM overlap samples: {sum(len(wm_val_map[v]) for v in common_values)}")
    print(f"UM overlap samples: {sum(len(um_val_map[v]) for v in common_values)}")

    ranked_values = sorted(
        common_values,
        key=lambda v: len(wm_val_map[v]) + len(um_val_map[v]),
        reverse=True,
    )

    print("\nTop common values (up to 30):")
    for value in ranked_values[:30]:
        print(
            f"  value={value}: WM={len(wm_val_map[value])} sample(s), "
            f"UM={len(um_val_map[value])} sample(s)"
        )

    print("\nExample pairs (up to 10):")
    shown = 0
    for value in ranked_values:
        for wm_sample in wm_val_map[value][:2]:
            for um_sample in um_val_map[value][:2]:
                print(f"  value={value}")
                print(f"    WM: {wm_sample.image_path}")
                print(f"    UM: {um_sample.image_path}")
                shown += 1
                if shown >= 30:
                    break
            if shown >= 30:
                break
        if shown >= 30:
            break

Cross-dataset overlap (value only, excluding 0)
Common values: 28
WM overlap samples: 28
UM overlap samples: 89

Top common values (up to 30):
  value=0.0: WM=1 sample(s), UM=58 sample(s)
  value=19.0: WM=1 sample(s), UM=3 sample(s)
  value=134.0: WM=1 sample(s), UM=3 sample(s)
  value=3.0: WM=1 sample(s), UM=1 sample(s)
  value=22.0: WM=1 sample(s), UM=1 sample(s)
  value=36.0: WM=1 sample(s), UM=1 sample(s)
  value=156.0: WM=1 sample(s), UM=1 sample(s)
  value=193.0: WM=1 sample(s), UM=1 sample(s)
  value=215.0: WM=1 sample(s), UM=1 sample(s)
  value=226.0: WM=1 sample(s), UM=1 sample(s)
  value=262.0: WM=1 sample(s), UM=1 sample(s)
  value=353.0: WM=1 sample(s), UM=1 sample(s)
  value=454.0: WM=1 sample(s), UM=1 sample(s)
  value=501.0: WM=1 sample(s), UM=1 sample(s)
  value=509.0: WM=1 sample(s), UM=1 sample(s)
  value=567.0: WM=1 sample(s), UM=1 sample(s)
  value=619.0: WM=1 sample(s), UM=1 sample(s)
  value=665.0: WM=1 sample(s), UM=1 sample(s)
  value=706.0: WM=1 sample(s), UM=1

### Cross-dataset duplicates: filename-pattern validation

- Method: matched samples across datasets by `value` only, excluded `value = 0`.
- Result: 27 common non-zero values, 27 WM samples in overlap.
- Filename pattern confirmation: 24/27 WM overlap samples (88.9%) have at least one UM counterpart whose filename contains the WM filename stem.
- Exception cases: 3 WM overlap sample(s) have no containment match inside same-value UM candidates (`3.0`, `22.0`, `226.0`).

Conclusion: filename containment is a strong heuristic for cross-dataset duplicates, but it is not complete and should be used together with value matching (and optionally visual similarity checks).
___

### Validate filename-pattern hypothesis for cross-dataset duplicates

In [39]:
# Validate filename-pattern hypothesis for cross-dataset duplicates
from collections import defaultdict

def value_key_nonzero(sample):
    if sample.value is None:
        return None
    value = round(float(sample.value), 3)
    if abs(value) < 1e-9:
        return None
    return value

um_all = um_train + um_valid + um_test

wm_val_map = defaultdict(list)
for sample in wm_samples:
    key = value_key_nonzero(sample)
    if key is not None:
        wm_val_map[key].append(sample)

um_val_map = defaultdict(list)
for sample in um_all:
    key = value_key_nonzero(sample)
    if key is not None:
        um_val_map[key].append(sample)

common_values = sorted(set(wm_val_map.keys()) & set(um_val_map.keys()))

containment_pairs = []
non_containment_pairs = []

for value in common_values:
    for wm_sample in wm_val_map[value]:
        wm_stem = wm_sample.image_path.stem.lower()
        matched_any = False
        for um_sample in um_val_map[value]:
            um_name = um_sample.image_path.name.lower()
            if wm_stem in um_name:
                containment_pairs.append((value, wm_sample, um_sample))
                matched_any = True
            else:
                non_containment_pairs.append((value, wm_sample, um_sample))
        if not matched_any:
            # Keep a marker when WM sample has no UM name containment among same-value samples
            containment_pairs.append((value, wm_sample, None))

# Aggregate stats
total_wm_overlap = sum(len(wm_val_map[v]) for v in common_values)
wm_with_containment = len({p[1].image_path for p in containment_pairs if p[2] is not None})
wm_without_containment = len({p[1].image_path for p in containment_pairs if p[2] is None})

print("=" * 78)
print("Filename containment hypothesis check")
print("Rule: for same value (excluding 0), UM filename contains WM filename stem")
print("=" * 78)
print(f"Common values: {len(common_values)}")
print(f"WM overlap samples: {total_wm_overlap}")
print(f"WM samples with >=1 containment match: {wm_with_containment}")
print(f"WM samples without containment match: {wm_without_containment}")

if total_wm_overlap > 0:
    ratio = wm_with_containment / total_wm_overlap
    print(f"Containment coverage: {wm_with_containment}/{total_wm_overlap} = {ratio:.1%}")

print("\nExamples of confirmed containment matches (up to 12):")
shown = 0
for value, wm_sample, um_sample in containment_pairs:
    if um_sample is None:
        continue
    print(f"  value={value}")
    print(f"    WM: {wm_sample.image_path.name}")
    print(f"    UM: {um_sample.image_path.name}")
    shown += 1
    if shown >= 12:
        break

missing_by_value = defaultdict(list)
for value, wm_sample, um_sample in containment_pairs:
    if um_sample is None:
        missing_by_value[value].append(wm_sample.image_path.name)

if missing_by_value:
    print("\nValues where containment was NOT found for some WM files:")
    for value in sorted(missing_by_value.keys()):
        uniq = sorted(set(missing_by_value[value]))
        print(f"  value={value}: {len(uniq)} WM file(s) without containment")
        for name in uniq[:5]:
            print(f"    - {name}")

report_block = []
report_block.append("### Cross-dataset duplicates: filename-pattern validation")
report_block.append("- Method: matched samples across datasets by `value` only, excluded `value = 0`.")
report_block.append(
    f"- Result: {len(common_values)} common non-zero values, "
    f"{total_wm_overlap} WM samples in overlap."
)
report_block.append(
    f"- Filename pattern confirmation: {wm_with_containment}/{total_wm_overlap} "
    f"WM overlap samples ({(wm_with_containment/total_wm_overlap):.1%}) have at least one UM counterpart "
    "whose filename contains the WM filename stem."
)
if wm_without_containment:
    report_block.append(
        f"- Exception cases: {wm_without_containment} WM overlap sample(s) have no containment match "
        "inside same-value UM candidates."
)
else:
    report_block.append("- No exception cases found for this rule.")

print("\n" + "=" * 78)
print("Report-ready summary")
print("=" * 78)
for line in report_block:
    print(line)

Filename containment hypothesis check
Rule: for same value (excluding 0), UM filename contains WM filename stem
Common values: 27
WM overlap samples: 27
WM samples with >=1 containment match: 24
WM samples without containment match: 3
Containment coverage: 24/27 = 88.9%

Examples of confirmed containment matches (up to 12):
  value=19.0
    WM: id_996_value_19_0.jpg
    UM: 0f6c7274-id_996_value_19_0_jpg.rf.a099670b026c51418cbf7d0ab07093a2.jpg
  value=36.0
    WM: id_251_value_36_0.jpg
    UM: 994205af-id_251_value_36_0_jpg.rf.b430b84117a964cb16bc22051001d74a.jpg
  value=134.0
    WM: id_488_value_134_0.jpg
    UM: 95f00d06-id_488_value_134_0_jpg.rf.f81da5bddd3de4b82e9f1fe84ac41f5a.jpg
  value=156.0
    WM: id_509_value_156_0.jpg
    UM: a800ac19-id_509_value_156_0_jpg.rf.3e916be8504d4ea1f004860f9d0420f0.jpg
  value=193.0
    WM: id_872_value_193_0.jpg
    UM: 617eac1a-id_872_value_193_0_jpg.rf.f0e4e0f57c18d30111a2d6436ab47ca9.jpg
  value=215.0
    WM: id_45_value_215_0.jpg
    UM: ce8

## Interactive Duplicate Removal

In [34]:

import pathlib

# Collect all duplicates from both datasets
wm_dups = find_dup_groups(wm_samples, dup_key)
um_dups = find_dup_groups(um_all, dup_key)
all_dups = wm_dups + um_dups

if not all_dups:
    print("No duplicates found.")
else:
    print(f"Total duplicate groups: {len(all_dups)}\n")
    
    # Load WM CSV for potential updates
    wm_csv_path = ROOT / "WaterMetricsDATA/waterMeterDataset/WaterMeters/data.csv"
    wm_df = pd.read_csv(wm_csv_path) if wm_csv_path.exists() else None
    
    for group_idx, group in enumerate(all_dups, 1):
        k = dup_key(group[0])
        print(f"{'='*70}")
        print(f"GROUP {group_idx} / {len(all_dups)}  |  val={k[0]}  size={k[1]} bytes")
        print(f"{'='*70}")
        
        # Show all paths
        for i, sample in enumerate(group):
            print(f"  [{i}]  {sample.image_path}")
        
        # Show images side-by-side
        fig, axes = plt.subplots(1, len(group), figsize=(5*len(group), 5))
        if len(group) == 1:
            axes = [axes]
        
        for idx, sample in enumerate(group):
            img = cv2.imread(str(sample.image_path))
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                axes[idx].imshow(img)
            axes[idx].set_title(f"[{idx}]")
            axes[idx].axis("off")
        
        plt.tight_layout()
        plt.show()
        
        # Ask user
        print(f"\nWhich index to KEEP? (0-{len(group)-1})")
        try:
            keep_idx = int(input(">>> "))
            if keep_idx < 0 or keep_idx >= len(group):
                print("Invalid index, skipping.")
                continue
        except ValueError:
            print("Skipped.")
            continue
        
        # Delete others
        deleted = []
        for i, sample in enumerate(group):
            if i != keep_idx:
                try:
                    img_path = sample.image_path
                    
                    # Delete image
                    img_path.unlink()
                    deleted.append(str(img_path))
                    
                    # Delete label file if exists (utility-meter)
                    label_path = img_path.parent.parent / "labels" / (img_path.stem + ".txt")
                    if label_path.exists():
                        label_path.unlink()
                        deleted.append(f"  + label: {label_path.name}")
                    
                    # Delete from CSV if waterMeterDataset
                    if sample.dataset_source == "water_meter_dataset" and wm_df is not None:
                        photo_name = img_path.name
                        wm_df = wm_df[wm_df["photo_name"] != photo_name]
                        deleted.append(f"  + removed from data.csv")
                    
                except Exception as e:
                    print(f"Error deleting {sample.image_path}: {e}")
        
        # Save updated CSV if modified
        if sample.dataset_source == "water_meter_dataset" and wm_df is not None:
            wm_df.to_csv(wm_csv_path, index=False)
        
        print(f"\nKept: [{keep_idx}]  {group[keep_idx].image_path}")
        for d in deleted:
            print(f"Deleted: {d}")
        print()
    
    # Summary
    if wm_df is not None and wm_csv_path.exists():
        print(f"\n{'='*70}")
        print(f"data.csv updated. New size: {len(wm_df)} rows")


No duplicates found.


## Выводы EDA

- **Размер датасетов:** waterMeter=1244, utility-meter=1939 (train+valid+test).
- **Распределение значений:** в waterMeter широкий диапазон показаний `0.000..99999.539`, среднее `422.507`; наблюдается длинный хвост редких больших значений.
- **Распределение классов цифр (utility-meter train):** всего 11755 digit-аннотаций; сильный дисбаланс по цифре `0` (3886) относительно остальных классов (`1..9`: ~759-1170).
- **Типичный размер ROI (waterMeter, нормализованные координаты):** медиана площади `0.0201`, диапазон `0.0027..0.1500` — ROI в среднем занимает небольшую часть кадра.
- **Пересечения между датасетами (по value, без 0):** 27 общих значений; 27 сэмплов WM и 31 сэмпл UM в зоне пересечения.
- **Паттерн дубликатов по имени файла:** в 24/27 случаев (88.9%) у UM-файла имя содержит stem соответствующего WM-файла; 3 исключения (`3.0`, `22.0`, `226.0`).
- **Потенциальные проблемы:**
  - перекос классов по цифре `0` может ухудшать качество OCR на редких цифрах;
  - наличие междатасетных дубликатов повышает риск data leakage при совместном обучении/валидации;
  - правило по имени файла полезно как эвристика, но не покрывает 100% случаев и должно дополняться проверкой по значению/визуальному сходству.

In [40]:
# Compact EDA summary metrics for report
from collections import Counter, defaultdict
from shapely.geometry import Polygon

# Dataset sizes
wm_n = len(wm_samples)
um_n = len(um_train) + len(um_valid) + len(um_test)

# Value stats (WM)
wm_values_local = [s.value for s in wm_samples if s.value is not None]
wm_min = min(wm_values_local) if wm_values_local else None
wm_max = max(wm_values_local) if wm_values_local else None
wm_mean = (sum(wm_values_local) / len(wm_values_local)) if wm_values_local else None

# Digit distribution (UM train)
all_digits_local = []
for s in um_train:
    if s.digit_bboxes:
        all_digits_local.extend([d[0] for d in s.digit_bboxes])
digit_counts_local = dict(sorted(Counter(all_digits_local).items()))

# ROI area stats (WM)
areas_local = []
for s in wm_samples:
    if s.roi_polygon and len(s.roi_polygon) >= 3:
        areas_local.append(Polygon(s.roi_polygon).area)
areas_local_sorted = sorted(areas_local)
roi_median = areas_local_sorted[len(areas_local_sorted)//2] if areas_local_sorted else None
roi_min = min(areas_local) if areas_local else None
roi_max = max(areas_local) if areas_local else None

# Overlap by value only (exclude 0)
def value_key_nonzero_local(sample):
    if sample.value is None:
        return None
    value = round(float(sample.value), 3)
    if abs(value) < 1e-9:
        return None
    return value

um_all_local = um_train + um_valid + um_test
wm_val_map_local = defaultdict(list)
for sample in wm_samples:
    key = value_key_nonzero_local(sample)
    if key is not None:
        wm_val_map_local[key].append(sample)

um_val_map_local = defaultdict(list)
for sample in um_all_local:
    key = value_key_nonzero_local(sample)
    if key is not None:
        um_val_map_local[key].append(sample)

common_values_local = sorted(set(wm_val_map_local.keys()) & set(um_val_map_local.keys()))
wm_overlap_local = sum(len(wm_val_map_local[v]) for v in common_values_local)
um_overlap_local = sum(len(um_val_map_local[v]) for v in common_values_local)

# Filename containment check
wm_with_containment_local = 0
for value in common_values_local:
    for wm_sample in wm_val_map_local[value]:
        wm_stem = wm_sample.image_path.stem.lower()
        has_match = any(wm_stem in um_sample.image_path.name.lower() for um_sample in um_val_map_local[value])
        if has_match:
            wm_with_containment_local += 1

wm_without_containment_local = wm_overlap_local - wm_with_containment_local
containment_ratio_local = (wm_with_containment_local / wm_overlap_local) if wm_overlap_local else 0.0

print(f"WM size: {wm_n}")
print(f"UM size (train+valid+test): {um_n}")
print(f"WM value range: {wm_min:.3f}..{wm_max:.3f}; mean={wm_mean:.3f}")
print(f"UM train total digit annotations: {len(all_digits_local)}")
print(f"UM train digit distribution: {digit_counts_local}")
print(f"WM ROI area (normalized): median={roi_median:.4f}, range={roi_min:.4f}..{roi_max:.4f}")
print(f"Cross-dataset overlap by value (excl. 0): common_values={len(common_values_local)}, WM_overlap={wm_overlap_local}, UM_overlap={um_overlap_local}")
print(f"Filename containment: {wm_with_containment_local}/{wm_overlap_local} ({containment_ratio_local:.1%}), no-match={wm_without_containment_local}")

WM size: 1244
UM size (train+valid+test): 1939
WM value range: 0.000..99999.539; mean=422.507
UM train total digit annotations: 11755
UM train digit distribution: {0: 3886, 1: 1170, 2: 973, 3: 918, 4: 848, 5: 845, 6: 825, 7: 772, 8: 759, 9: 759}
WM ROI area (normalized): median=0.0201, range=0.0027..0.1500
Cross-dataset overlap by value (excl. 0): common_values=27, WM_overlap=27, UM_overlap=31
Filename containment: 24/27 (88.9%), no-match=3
